# Playground Series S6E7 — Predicting Student Health Risk
## Score: .94864

In [8]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore")

SEED = 42
N_SPLITS = 5
DATA_DIR = Path("playground-series-s6e7")
TARGET = "health_condition"
ID_COL = "id"

rng = np.random.default_rng(SEED)
print("LightGBM", lgb.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

LightGBM 4.6.0 | pandas 3.0.3 | numpy 2.4.6


In [9]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("train:", train.shape, "| test:", test.shape)

CAT_COLS = ["diet_type", "stress_level", "sleep_quality",
            "physical_activity_level", "smoking_alcohol", "gender"]
NUM_COLS = ["sleep_duration", "heart_rate", "bmi", "calorie_expenditure",
            "step_count", "exercise_duration", "water_intake"]

CLASSES = ["at-risk", "unhealthy", "fit"]
class_to_int = {c: i for i, c in enumerate(CLASSES)}
int_to_class = {i: c for c, i in class_to_int.items()}
y = train[TARGET].map(class_to_int).to_numpy()

print("\nTarget distribution:")
print(train[TARGET].value_counts(normalize=True).round(4))

train: (690088, 15) | test: (295753, 14)

Target distribution:
health_condition
at-risk      0.8587
unhealthy    0.0836
fit          0.0577
Name: proportion, dtype: float64


## Feature preparation

In [10]:
EPS = 1e-3
MISS_FLAGS = [f"{c}_isna" for c in NUM_COLS + CAT_COLS]
ENG_NUM = [
    "steps_per_cal", "cal_per_min_exercise", "steps_per_min_exercise", "cal_per_step",
    "hr_per_bmi", "cal_per_bmi", "water_per_bmi",
    "steps_per_sleep", "exercise_per_sleep", "cal_per_sleep",
]
ENG_CAT = ["bmi_cat", "sleep_cat", "stress_x_activity", "diet_x_smoke"]
CAT_FEATURES = CAT_COLS + ENG_CAT


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    X = df.copy()

    for c in NUM_COLS + CAT_COLS:
        X[f"{c}_isna"] = df[c].isna().astype("int8")
    X["n_missing"] = df[NUM_COLS + CAT_COLS].isna().sum(axis=1)

    X["steps_per_cal"] = X["step_count"] / (X["calorie_expenditure"] + EPS)
    X["cal_per_min_exercise"] = X["calorie_expenditure"] / (X["exercise_duration"] + EPS)
    X["steps_per_min_exercise"] = X["step_count"] / (X["exercise_duration"] + EPS)
    X["cal_per_step"] = X["calorie_expenditure"] / (X["step_count"] + EPS)
    X["hr_per_bmi"] = X["heart_rate"] / (X["bmi"] + EPS)
    X["cal_per_bmi"] = X["calorie_expenditure"] / (X["bmi"] + EPS)
    X["water_per_bmi"] = X["water_intake"] / (X["bmi"] + EPS)
    X["steps_per_sleep"] = X["step_count"] / (X["sleep_duration"] + EPS)
    X["exercise_per_sleep"] = X["exercise_duration"] / (X["sleep_duration"] + EPS)
    X["cal_per_sleep"] = X["calorie_expenditure"] / (X["sleep_duration"] + EPS)

    X["bmi_cat"] = pd.cut(X["bmi"], bins=[-np.inf, 18.5, 25, 30, np.inf],
                          labels=["under", "normal", "over", "obese"]).astype("object")
    X["sleep_cat"] = pd.cut(X["sleep_duration"], bins=[-np.inf, 6, 8, np.inf],
                            labels=["short", "normal", "long"]).astype("object")

    def combo(a, b):
        return df[a].fillna("NA").astype(str) + "|" + df[b].fillna("NA").astype(str)
    X["stress_x_activity"] = combo("stress_level", "physical_activity_level")
    X["diet_x_smoke"] = combo("diet_type", "smoking_alcohol")

    for c in CAT_FEATURES:
        X[c] = X[c].fillna("__NA__").astype("category")

    return X


FEATURES = NUM_COLS + ENG_NUM + MISS_FLAGS + ["n_missing"] + CAT_FEATURES

X = build_features(train)[FEATURES]
X_test = build_features(test)[FEATURES]

for c in CAT_FEATURES:
    levels = X[c].cat.categories.union(X_test[c].cat.categories)
    X[c] = X[c].cat.set_categories(levels)
    X_test[c] = X_test[c].cat.set_categories(levels)

print("Features:", len(FEATURES))
print("Categorical:", CAT_FEATURES)

Features: 41
Categorical: ['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender', 'bmi_cat', 'sleep_cat', 'stress_x_activity', 'diet_x_smoke']


## Cross-validated training

In [11]:
lgb_params = dict(
    objective="multiclass",
    num_class=len(CLASSES),
    n_estimators=6000,
    learning_rate=0.02,
    num_leaves=96,
    max_depth=-1,
    min_child_samples=200,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.7,
    reg_alpha=0.5,
    reg_lambda=5.0,
    min_split_gain=1e-3,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1,
    verbose=-1,
)

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof_proba = np.zeros((len(X), len(CLASSES)))
test_proba = np.zeros((len(X_test), len(CLASSES)))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        X.iloc[tr_idx], y[tr_idx],
        eval_set=[(X.iloc[va_idx], y[va_idx])],
        eval_metric="multi_logloss",
        categorical_feature=CAT_FEATURES,
        callbacks=[lgb.early_stopping(200), lgb.log_evaluation(0)],
    )
    oof_proba[va_idx] = model.predict_proba(X.iloc[va_idx])
    test_proba += model.predict_proba(X_test) / N_SPLITS

    fold_pred = oof_proba[va_idx].argmax(axis=1)
    fold_bal = balanced_accuracy_score(y[va_idx], fold_pred)
    print(f"fold {fold}: best_iter={model.best_iteration_:>4} | argmax bal-acc={fold_bal:.5f}")

argmax_oof = oof_proba.argmax(axis=1)
print(f"\nOOF balanced accuracy (plain argmax): {balanced_accuracy_score(y, argmax_oof):.5f}")

Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[5227]	valid_0's multi_logloss: 0.106816
fold 0: best_iter=5227 | argmax bal-acc=0.92327
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[5295]	valid_0's multi_logloss: 0.10697
fold 1: best_iter=5295 | argmax bal-acc=0.92468
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[5116]	valid_0's multi_logloss: 0.111305
fold 2: best_iter=5116 | argmax bal-acc=0.92512
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[5378]	valid_0's multi_logloss: 0.107501
fold 3: best_iter=5378 | argmax bal-acc=0.92235
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[5065]	valid_0's multi_logloss: 0.111365
fold 4: best_iter=5065 | argmax bal-acc=0.92324

OOF balanced accuracy (plain argmax): 0.92373


## Tune the decision rule for balanced accuracy

In [12]:
def weighted_predict(proba: np.ndarray, w: np.ndarray) -> np.ndarray:
    return (proba * w).argmax(axis=1)


def search_weights(proba, y_true, w_grid_u, w_grid_f, base=None):
    best_w = np.array([1.0, 1.0, 1.0]) if base is None else base.copy()
    best_score = balanced_accuracy_score(y_true, weighted_predict(proba, best_w))
    for wu in w_grid_u:
        for wf in w_grid_f:
            w = np.array([1.0, wu, wf])
            score = balanced_accuracy_score(y_true, weighted_predict(proba, w))
            if score > best_score:
                best_score, best_w = score, w
    return best_w, best_score


coarse = np.round(np.geomspace(0.5, 20, 22), 3)
best_w, best_score = search_weights(oof_proba, y, coarse, coarse)

fine_u = np.linspace(best_w[1] * 0.6, best_w[1] * 1.6, 21)
fine_f = np.linspace(best_w[2] * 0.6, best_w[2] * 1.6, 21)
best_w, best_score = search_weights(oof_proba, y, fine_u, fine_f, base=best_w)

print(f"Best weights (at-risk, unhealthy, fit): {best_w.round(3)}")
print(f"OOF balanced accuracy (tuned):   {best_score:.5f}")
print(f"OOF balanced accuracy (argmax):  {balanced_accuracy_score(y, argmax_oof):.5f}")

tuned_oof = weighted_predict(oof_proba, best_w)
print("\nConfusion matrix (rows=true, cols=pred) [at-risk, unhealthy, fit]:")
print(confusion_matrix(y, tuned_oof))
print("\n", classification_report(y, tuned_oof, target_names=CLASSES, digits=4))

Best weights (at-risk, unhealthy, fit): [ 1.    14.075 14.075]
OOF balanced accuracy (tuned):   0.94835
OOF balanced accuracy (argmax):  0.92373

Confusion matrix (rows=true, cols=pred) [at-risk, unhealthy, fit]:
[[552136  26511  13914]
 [  1736  55783    205]
 [  1846    268  37689]]

               precision    recall  f1-score   support

     at-risk     0.9936    0.9318    0.9617    592561
   unhealthy     0.6756    0.9664    0.7953     57724
         fit     0.7275    0.9469    0.8228     39803

    accuracy                         0.9355    690088
   macro avg     0.7989    0.9483    0.8599    690088
weighted avg     0.9516    0.9355    0.9397    690088



## Predict test set & write submission

In [13]:
test_pred_int = weighted_predict(test_proba, best_w)
test_pred_lbl = pd.Series(test_pred_int).map(int_to_class)

submission = pd.DataFrame({ID_COL: test[ID_COL], TARGET: test_pred_lbl})
submission.to_csv("submission.csv", index=False)

print("Wrote submission.csv:", submission.shape)
print("\nPredicted label distribution:")
print(submission[TARGET].value_counts(normalize=True).round(4))
print("\nTrain label distribution (for reference):")
print(train[TARGET].value_counts(normalize=True).round(4))
submission.head()

Wrote submission.csv: (295753, 2)

Predicted label distribution:
health_condition
at-risk      0.8065
unhealthy    0.1198
fit          0.0737
Name: proportion, dtype: float64

Train label distribution (for reference):
health_condition
at-risk      0.8587
unhealthy    0.0836
fit          0.0577
Name: proportion, dtype: float64


,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy
